In [1]:
import feast
import os
from loguru import logger
import dagshub
import mlflow
import time
import joblib
from datetime import datetime
import uuid
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from feast import (
    FeatureStore,
    Entity,
    FeatureService,
    FeatureView,
    Field,
    FileSource
)
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from feast.types import Float32, Float64, Int64, String

import reescalador

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.model_selection import (GridSearchCV,
                                     RandomizedSearchCV,
                                     train_test_split)
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (classification_report,
                            confusion_matrix,
                            ConfusionMatrixDisplay,
                            accuracy_score)
from sklearn import svm
from xgboost import XGBClassifier, plot_importance
from catboost import CatBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier


dagshub.init(repo_owner='Marquinhos9873', repo_name='mle3', mlflow=True)


Accessing as Marquinhos9873

Initialized MLflow to track repo "Marquinhos9873/mle3"

Repository Marquinhos9873/mle3 initialized!

In [3]:
salaries = pd.read_csv("../data/Dataset salary 2024.csv")

In [4]:
salaries

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2024,SE,FT,AI Engineer,202730,USD,202730,US,0,US,M
1,2024,SE,FT,AI Engineer,92118,USD,92118,US,0,US,M
2,2024,SE,FT,Data Engineer,130500,USD,130500,US,0,US,M
3,2024,SE,FT,Data Engineer,96000,USD,96000,US,0,US,M
4,2024,SE,FT,Machine Learning Engineer,190000,USD,190000,US,0,US,M
...,...,...,...,...,...,...,...,...,...,...,...
16529,2020,SE,FT,Data Scientist,412000,USD,412000,US,100,US,L
16530,2021,MI,FT,Principal Data Scientist,151000,USD,151000,US,100,US,L
16531,2020,EN,FT,Data Scientist,105000,USD,105000,US,100,US,S
16532,2020,EN,CT,Business Data Analyst,100000,USD,100000,US,100,US,L


#### FEATURE STORE

In [10]:
class Feast_use():
    def __init__(self, filepath, external_data=None):
        self.filepath = filepath
        self.feature_table = None
        self.external_data = external_data

        
    def table_creator(self, feature_table=None, external_data=None):

       
        if feature_table is None or feature_table.empty:
            feature_table = pd.DataFrame()
        
        self.feature_table = feature_table

        
        self.feature_table['UId'] = [str(uuid.uuid4()) for _ in range(self.feature_table.shape[0])]
        self.feature_table["created"] = [datetime.now() for _ in range(self.feature_table.shape[0])]
        logger.info(f"Feature Table creada -> {datetime.now()}")

       
        if external_data is not None:
            self.feature_table = pd.concat(
                [
                    self.feature_table.reset_index(drop=True),
                    external_data.reset_index(drop=True)
                ],
                axis=1
            )

        return self.feature_table


    def write_feature_table(self, filepath: str) -> None:
        if self.feature_table is not None and not self.feature_table.empty:
            self.feature_table.to_parquet(f"{filepath}.parquet", index=False)
            self.feature_table.to_csv(f"{filepath}.csv", index=False)
        else:
            raise Exception("La feature table no ha sido creada.")


f = Feast_use(filepath= '/home/marcoloq/mle2/Feature_Store_3/valued_pup/feature_repo')
feature_table_new = f.table_creator(external_data=reducing_data)
logger.info("Guardando feature table")
f.write_feature_table(filepath='/home/marcoloq/mle2/Feature_Store_3/valued_pup/feature_repo')


2025-11-18 21:06:12.021 | INFO     | __main__:table_creator:19 - Feature Table creada -> 2025-11-18 21:06:12.021518
2025-11-18 21:06:12.026 | INFO     | __main__:<module>:44 - Guardando feature table


#### Set tracking // Params

In [13]:
params={"Decision Tree": {
                        "criterion": ['gini','log_loss'],
                        "max_depth": [0, 10, 12, 16],
                        "min_samples_split": [2, 3, 4, 5]
                    },
    
                     "Random Forest":{
                        "n_estimators" : [110 , 115, 125, 130],
                        "criterion" : ['gini' ,'log_loss', 'entropy'],
                        "max_depth" : [None, 5, 7 , 8]
                    },
    
                    "Gradient Boosting": {
                        "n_estimators" : [110 , 115, 125, 130],
                        "max_depth" : [None, 5, 7 , 8],
                        "learning_rate": [0.0045 , 0.01, 0.05, 0.10],
                        "min_samples_split": [2, 3, 4, 5],
                    },
                    "XGBClassifier":{
                        'learning_rate':[0.0045 , 0.01, 0.05, 0.10],
                        'max_depth': [5, 6, 7],
                        'n_estimators': [256, 128, 64, 12],
                        'tree_method': ['auto', 'approx']
                    },
    
                    "CatBoosting Classifier":{
                        'iterations': [200, 400, 800],
                        'learning_rate': [0.001, 0.01, 0.05, 0.1],
                        'depth': [3, 4, 6, 8, 10],
                        'l2_leaf_reg': [1, 3, 5, 7, 9],
                        'bootstrap_type': ['Bayesian', 'Bernoulli', 'MVS']
                    }
                    
                }

In [6]:
SAVE_DIR = "/home/marcoloq/mle2/data/model_metrics_local_pureba_3/"
os.makedirs(SAVE_DIR, exist_ok=True)


In [15]:
xgb = XGBClassifier(device= 'cuda', verbosity = 1)

In [16]:
kitty = CatBoostClassifier(task_type = 'GPU', early_stopping_rounds = 100)

In [17]:
Forest = RandomForestClassifier(verbose = 1)

In [18]:
Grades = GradientBoostingClassifier(verbose = 1) 

In [7]:
Ada = AdaBoostClassifier()

In [10]:
machine = SVC()

#### Def calculate_clsf_model

In [67]:
def evaluate_classification_model(name_model: str, y_real, predictions, probabilities, save_path: str = None):
    
    
    label_map = {
        0: "No Stress",
        1: "Distress",
        2: "Eustress"
    }
    display_labels = [label_map[i] for i in sorted(label_map.keys())]

    
    cm = confusion_matrix(y_real, predictions)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=display_labels)

    plt.figure(figsize=(6, 5))
    ax = plt.gca()                     
    disp.plot(cmap="Blues", values_format='d', ax=ax)
    plt.title(f"Matriz de Confusión - {name_model}")
    plt.tight_layout()

    if save_path:
        plt.savefig(f"{save_path}/{name_model}_confusion_matrix.png")

    plt.show()


    
    accuracy = accuracy_score(y_real, predictions)
    precision = precision_score(y_real, predictions, average='weighted')
    recall = recall_score(y_real, predictions, average='weighted')
    f1 = f1_score(y_real, predictions, average='weighted')

    
    try:
        auc_score = roc_auc_score(y_real, probabilities, multi_class='ovr')
    except:
        auc_score = None
        print("AUC no disponible para este modelo.")

    print(f"\n>>> Métricas del modelo: {name_model}")
    print(f"accuracy : {accuracy}")
    print(f"precision: {precision}")
    print(f"recall   : {recall}")
    print(f"f1_score : {f1}")
    print(f"auc_score: {auc_score}")

    metricas = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "auc_score": auc_score
    }



    return metricas
